<h1><strong><span style="font-size: x-large;">ESTUDIO GRAFO STAR WARS.</span></strong></h1>

<p><h3>Construcción grafo Star Wars</h3></p>

In [52]:
import csv
def parse_grafo(ruta):
    g=[]
    with open(ruta , encoding="utf-8") as f:
        lector= csv.reader(f)
        next(lector) 
        for linea in lector:
            relacion=tuple()
            relacion=(linea[0], linea[1], int(linea[2]))
            g.append(relacion)
        return Graph(g, weighted=True)
        

In [53]:
g_ep1= parse_grafo("starwars_ep1_edges_clean.csv")
g_ep2= parse_grafo("starwars_ep2_edges_clean.csv")
g_ep3= parse_grafo("starwars_ep3_edges_clean.csv")
g_ep4= parse_grafo("starwars_ep4_edges_clean.csv")
g_ep5= parse_grafo("starwars_ep5_edges_clean.csv")
g_ep6= parse_grafo("starwars_ep6_edges_clean.csv")
g_ep7= parse_grafo("starwars_ep7_edges_clean.csv")
g_general= parse_grafo("starwars_full_interactions_edges_clean.csv")

In [54]:
import networkx as nx

G_ep1 = nx.Graph()
for a, b, peso in g_ep1.edges():
    G_ep1.add_edge(a, b, weight=peso)
    
G_ep2 = nx.Graph()
for a, b, peso in g_ep2.edges():
    G_ep2.add_edge(a, b, weight=peso)
    
G_ep3 = nx.Graph()
for a, b, peso in g_ep3.edges():
    G_ep3.add_edge(a, b, weight=peso)
    
G_ep4 = nx.Graph()
for a, b, peso in g_ep4.edges():
    G_ep4.add_edge(a, b, weight=peso)
    
G_ep5 = nx.Graph()
for a, b, peso in g_ep5.edges():
    G_ep5.add_edge(a, b, weight=peso)
        
G_ep6 = nx.Graph()
for a, b, peso in g_ep6.edges():
    G_ep6.add_edge(a, b, weight=peso)
    
G_ep7 = nx.Graph()
for a, b, peso in g_ep7.edges():
    G_ep7.add_edge(a, b, weight=peso)
    
G_general = nx.Graph()
for a, b, peso in g_general.edges():
    G_general.add_edge(a, b, weight=peso)

<h2>Metodología</h2>

<h3>Construcción de los grafos</h3>
<p>Los datos proceden del dataset de Evelina Gabasova sobre las interacciones entre personajes de Star Wars. Cada personaje es un nodo y se crea una arista entre dos personajes cuando ambos pronuncian al menos una línea de diálogo en la misma escena. El peso de la arista es el número de escenas en las que coinciden. Se ha construido un grafo no dirigido y ponderado para cada episodio (I a VII) y otro con la red completa de la saga.</p>

<h3>Métricas empleadas</h3>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Métrica</th><th>Qué mide</th><th>Interpretación en la red</th></tr>
  <tr><td>Grado</td><td>Número de personajes distintos con los que comparte escena</td><td>Cantidad de contactos</td></tr>
  <tr><td>Grado ponderado</td><td>Suma de los pesos de sus aristas</td><td>Número total de escenas compartidas con otros personajes</td></tr>
  <tr><td>Intermediación (betweenness)</td><td>Proporción de caminos mínimos entre pares de personajes que pasan por el nodo</td><td>Personajes que actúan de puente entre grupos</td></tr>
  <tr><td>Autovector (eigenvector)</td><td>Importancia de un nodo en función de la importancia de sus vecinos</td><td>Estar conectado con los personajes importantes, no solo con muchos</td></tr>
  <tr><td>Cercanía (closeness)</td><td>Inverso de la distancia media al resto de nodos</td><td>Rapidez con la que un personaje llega a todos los demás</td></tr>
</table>

<p>La intermediación y la cercanía se calculan sin pesos, de modo que reflejan únicamente la estructura de la red. La razón es que en este grafo un peso alto indica mayor relación (más cercanía), mientras que estos algoritmos interpretan el peso como una distancia. El grado ponderado y el autovector sí utilizan los pesos.</p>

<p>Para describir la estructura global se calculan el número de nodos y aristas, la densidad, las componentes conexas, el camino medio y el diámetro. Las comunidades se detectan con el algoritmo de maximización voraz de la modularidad (greedy modularity), usando los pesos de las aristas.</p>

<h3>Limitaciones del método</h3>
<ul>
  <li><b>Sesgo de reunión:</b> al crear las aristas por escena, si varios personajes hablan en la misma escena todos quedan conectados entre sí. Esto infla el grado de los personajes secundarios que aparecen en escenas grupales, por lo que el grado se contrasta siempre con el grado ponderado.</li>
  <li><b>Menciones indirectas:</b> si un personaje habla de otro que no está presente en la escena, no se crea ninguna arista.</li>
<li><b>Nombres de personajes:</b> un mismo personaje puede aparecer con nombres distintos según la etapa de la saga. En los grafos de cada episodio, ANAKIN y DARTH VADER son nodos separados (en el episodio VI aparecen los dos), mientras que en el grafo general aparecen unificados en DARTH VADER.</li>

<h3>Análisis grafo episodio 1</h3>

<p>Grado</p>

In [55]:
grado = dict(G_ep1.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('QUI-GON', 26), ('ANAKIN', 23), ('JAR JAR', 18), ('PADME', 18), ('OBI-WAN', 13), ('EMPEROR', 11), ('NUTE GUNRAY', 10), ('CAPTAIN PANAKA', 9), ('KITSTER', 9), ('SHMI', 8)]


<p>Grado ponderado.</p>

In [56]:
grado_pond = dict(G_ep1.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('QUI-GON', 151), ('ANAKIN', 111), ('PADME', 81), ('JAR JAR', 78), ('OBI-WAN', 65), ('R2-D2', 48), ('CAPTAIN PANAKA', 41), ('SHMI', 27), ('NUTE GUNRAY', 24), ('EMPEROR', 23)]


<p>Intermediación (betweenness)</p>

In [57]:
inter = nx.betweenness_centrality(G_ep1)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('QUI-GON', 0.3226135226135226), ('ANAKIN', 0.16988362821696154), ('JAR JAR', 0.14583615000281666), ('NUTE GUNRAY', 0.1254313837647171), ('PADME', 0.10342138675472011), ('EMPEROR', 0.10023551690218359), ('TC-14', 0.07318628151961486), ('OBI-WAN', 0.04407145240478574), ('RIC OLIE', 0.02399239065905733), ('VALORUM', 0.015785265785265786)]


<p>Autovector (eigenvector)</p>

In [58]:
auto = nx.eigenvector_centrality(G_ep1, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('QUI-GON', 0.5541879173449947), ('ANAKIN', 0.41731372617986107), ('JAR JAR', 0.3515583921104325), ('PADME', 0.3500400944801737), ('OBI-WAN', 0.3267332138652381), ('R2-D2', 0.26477353836686746), ('CAPTAIN PANAKA', 0.18919348729693705), ('SHMI', 0.14969773069443834), ('WATTO', 0.09783112812608905), ('RIC OLIE', 0.064383078867805)]


<p>Cercanía (closeness)</p>

In [59]:
cerc = nx.closeness_centrality(G_ep1)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('QUI-GON', 0.7708333333333334), ('ANAKIN', 0.6491228070175439), ('PADME', 0.6491228070175439), ('JAR JAR', 0.6271186440677966), ('OBI-WAN', 0.5692307692307692), ('EMPEROR', 0.5606060606060606), ('NUTE GUNRAY', 0.5522388059701493), ('CAPTAIN PANAKA', 0.5441176470588235), ('SIO BIBBLE', 0.5285714285714286), ('KITSTER', 0.5211267605633803)]


<p>Comunidades</p>

In [60]:
comunidades = nx.community.greedy_modularity_communities(G_ep1, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

11
['TEY HOW', 'DOFINE', 'SIO BIBBLE', 'PK-4', 'NUTE GUNRAY', 'BAIL ORGANA', 'RUNE', 'VALORUM', 'DARTH MAUL', 'TC-14', 'EMPEROR', 'GENERAL CEEL']
['C-3PO', 'GREEDO', 'WALD', 'R2-D2', 'WATTO', 'KITSTER']
['JAR JAR', 'FODE/BEED', 'SEBULBA', 'JABBA', 'TARPALS']
['RABE', 'YODA', 'MACE WINDU', 'KI-ADI-MUNDI']
['RIC OLIE', 'BRAVO THREE', 'BRAVO TWO']
['BOSS NASS', 'CAPTAIN PANAKA']
['PADME', 'JIRA']
['ANAKIN']
['OBI-WAN']
['QUI-GON']
['SHMI']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [61]:
print(
    f"nodos: {G_ep1.number_of_nodes()} aristas: {G_ep1.number_of_edges()}"
    f" densidad: {nx.density(G_ep1)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep1)}")

nodos: 38 aristas: 135 densidad: 0.19203413940256045
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [62]:
print(nx.average_shortest_path_length(G_ep1))
print(nx.diameter(G_ep1))

2.1052631578947367
4


<h3>Resultados del episodio 1</h3>

<p><b>Estructura.</b> La red tiene 38 personajes y 135 aristas, con una densidad de 0,19. Es un grafo conexo (una sola componente), con un camino medio de 2,11 y un diámetro de 4. Es decir, la red es compacta: desde cualquier personaje se llega a cualquier otro en un máximo de 4 pasos y, en promedio, en poco más de 2.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Qui-Gon (26)</td><td>Qui-Gon (151)</td><td>Qui-Gon (0,323)</td><td>Qui-Gon (0,554)</td><td>Qui-Gon (0,771)</td></tr>
  <tr><td>2</td><td>Anakin (23)</td><td>Anakin (111)</td><td>Anakin (0,170)</td><td>Anakin (0,417)</td><td>Anakin (0,649)</td></tr>
  <tr><td>3</td><td>Jar Jar (18)</td><td>Padmé (81)</td><td>Jar Jar (0,146)</td><td>Jar Jar (0,352)</td><td>Padmé (0,649)</td></tr>
  <tr><td>4</td><td>Padmé (18)</td><td>Jar Jar (78)</td><td>Nute Gunray (0,125)</td><td>Padmé (0,350)</td><td>Jar Jar (0,627)</td></tr>
  <tr><td>5</td><td>Obi-Wan (13)</td><td>Obi-Wan (65)</td><td>Padmé (0,103)</td><td>Obi-Wan (0,327)</td><td>Obi-Wan (0,569)</td></tr>
</table>

<p><b>El protagonista.</b> Qui-Gon encabeza las cinco métricas y Anakin es siempre el segundo, de modo que el resultado no depende de la métrica elegida. Qui-Gon comparte escena con 26 de los otros 37 personajes (un 70 %) y acumula 151 escenas compartidas, lo que concuerda con su papel de guía de la acción en este episodio.</p>

<p><b>Grado frente a grado ponderado.</b> Los dos rankings coinciden en los primeros puestos, pero aparecen diferencias que ilustran el sesgo de reunión. El Emperador (11 contactos, sexto por grado) cae al décimo puesto en grado ponderado (23 escenas), y Kitster (9 contactos) ni siquiera entra en el top 10 ponderado. Son personajes con muchos contactos pero con pocas escenas con cada uno, es decir, conexiones generadas por escenas grupales. En sentido contrario, R2-D2 es sexto por grado ponderado (48) y no está entre los diez primeros por grado: habla con pocos personajes, pero comparte muchas escenas con ellos.</p>

<p><b>Puentes entre grupos.</b> En intermediación aparecen personajes que no destacan en las demás métricas: Nute Gunray (cuarto), el Emperador (sexto), TC-14 (séptimo) y Valorum (décimo). Son personajes del bando de la Federación de Comercio y del Senado que conectan este grupo con el núcleo principal de la historia. En el caso de TC-14, además, es el enlace de un personaje periférico (PK-4) con el resto de la red.</p>

<p><b>El papel del Emperador.</b> En el episodio I el Emperador (Palpatine) ocupa el sexto puesto en intermediación (0,100), por detrás de Qui-Gon, Anakin, Jar Jar, Nute Gunray y Padmé. Por tanto, en este episodio es un puente relevante, pero no el principal. Su papel como conector entre facciones se contrasta en los episodios II y III.</p>

<p><b>Comunidades.</b> El algoritmo detecta 11 comunidades, de las que se pueden interpretar las principales:</p>
<ul>
  <li><b>Federación de Comercio y Senado:</b> Valorum, Nute Gunray, Bail Organa, Emperador, Darth Maul, Tey How, Rune, Dofine, General Ceel, TC-14, Sio Bibble y PK-4.</li>
  <li><b>Tatooine:</b> R2-D2, Kitster, C-3PO, Watto, Greedo y Wald.</li>
  <li><b>Carrera de vainas y gungans:</b> Jabba, Fode/Beed, Sebulba, Jar Jar y Tarpals.</li>
  <li><b>Consejo Jedi:</b> Yoda, Mace Windu y Ki-Adi-Mundi (junto con Rabe).</li>
  <li><b>Pilotos de Naboo:</b> Bravo Two, Bravo Three y Ric Olie.</li>
  <li>Otros grupos pequeños: Boss Nass y Capitán Panaka; Jira y Padmé.</li>
</ul>
<p>Qui-Gon, Anakin, Obi-Wan y Shmi quedan como comunidades de un solo elemento. En el caso de los tres protagonistas, la razón es que sus conexiones se reparten entre muchos grupos distintos, por lo que el algoritmo no puede asignarlos a ninguno. Las comunidades resultantes coinciden en general con los escenarios y facciones de la trama. Sin utilizar los pesos, el algoritmo devuelve una única comunidad con todos los personajes, por lo que se ha empleado la versión ponderada.</p>

<h3>Análisis grafo episodio 2</h3>

<p>Grado</p>

In [63]:
grado = dict(G_ep2.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 21), ('OBI-WAN', 18), ('PADME', 17), ('COUNT DOOKU', 10), ('JAR JAR', 10), ('MACE WINDU', 10), ('YODA', 10), ('EMPEROR', 9), ('BAIL ORGANA', 7), ('SENATOR ASK AAK', 7)]


<p>Grado ponderado.</p>

In [64]:
grado_pond = dict(G_ep2.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 65), ('OBI-WAN', 51), ('PADME', 50), ('MACE WINDU', 24), ('YODA', 24), ('R2-D2', 17), ('COUNT DOOKU', 16), ('C-3PO', 15), ('JAR JAR', 15), ('EMPEROR', 15)]


<p>Intermediación (betweenness)</p>

In [65]:
inter = nx.betweenness_centrality(G_ep2)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 0.3833822091886608), ('OBI-WAN', 0.3148010927244798), ('PADME', 0.178946725317693), ('KI-ADI-MUNDI', 0.0625), ('JAR JAR', 0.06213037634408602), ('MACE WINDU', 0.06106165340036306), ('YODA', 0.06106165340036306), ('COUNT DOOKU', 0.03698287599497276), ('EMPEROR', 0.028351487222454962), ('BAIL ORGANA', 0.009538646837033935)]


<p>Autovector (eigenvector)</p>

In [66]:
auto = nx.eigenvector_centrality(G_ep2, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 0.5794937843088441), ('PADME', 0.49033454715273406), ('OBI-WAN', 0.39487909782212144), ('R2-D2', 0.262763938066438), ('C-3PO', 0.17673428498536736), ('MACE WINDU', 0.16788378479100485), ('YODA', 0.16540557715134396), ('COUNT DOOKU', 0.14235135597528606), ('OWEN', 0.14007848040420015), ('JAR JAR', 0.13139560639631453)]


<p>Cercanía (closeness)</p>

In [67]:
cerc = nx.closeness_centrality(G_ep2)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 0.7272727272727273), ('OBI-WAN', 0.6808510638297872), ('PADME', 0.6666666666666666), ('MACE WINDU', 0.5925925925925926), ('YODA', 0.5925925925925926), ('COUNT DOOKU', 0.5818181818181818), ('JAR JAR', 0.5818181818181818), ('NUTE GUNRAY', 0.5161290322580645), ('POGGLE', 0.5161290322580645), ('SUN RIT', 0.5161290322580645)]


<p>Comunidades</p>

In [68]:
comunidades = nx.community.greedy_modularity_communities(G_ep2, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

8
['ORN FREE TAA', 'MACE WINDU', 'JAR JAR', 'KI-ADI-MUNDI', 'PLO KOON', 'BAIL ORGANA', 'SENATOR ASK AAK', 'YODA', 'EMPEROR']
['BERU', 'OWEN', 'C-3PO', 'CLIEGG']
['PADME', 'SOLA', 'JOBAL', 'RUWEE']
['SUN RIT', 'COUNT DOOKU', 'POGGLE', 'NUTE GUNRAY']
['SIO BIBBLE', 'WATTO', 'SHMI', 'ANAKIN']
['LAMA SU', 'TAUN WE', 'BOBA FETT', 'JANGO FETT']
['PK-4', 'CAPTAIN TYPHO', 'OBI-WAN']
['R2-D2']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [69]:
print(
    f"nodos: {G_ep2.number_of_nodes()} aristas: {G_ep2.number_of_edges()}"
    f" densidad: {nx.density(G_ep2)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep2)}")

nodos: 33 aristas: 101 densidad: 0.19128787878787878
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [70]:
print(nx.average_shortest_path_length(G_ep2))
print(nx.diameter(G_ep2))

2.1382575757575757
4


<h3>Resultados del episodio 2</h3>

<p><b>Estructura.</b> La red tiene 33 personajes y 101 aristas, con una densidad de 0,19. Es un grafo conexo (una sola componente), con un camino medio de 2,14 y un diámetro de 4. La estructura es prácticamente idéntica a la del episodio I (densidad de 0,19, diámetro de 4 y camino medio de 2,11): una red compacta en la que desde cualquier personaje se llega a cualquier otro en un máximo de 4 pasos.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Anakin (21)</td><td>Anakin (65)</td><td>Anakin (0,383)</td><td>Anakin (0,579)</td><td>Anakin (0,727)</td></tr>
  <tr><td>2</td><td>Obi-Wan (18)</td><td>Obi-Wan (51)</td><td>Obi-Wan (0,315)</td><td>Padmé (0,490)</td><td>Obi-Wan (0,681)</td></tr>
  <tr><td>3</td><td>Padmé (17)</td><td>Padmé (50)</td><td>Padmé (0,179)</td><td>Obi-Wan (0,395)</td><td>Padmé (0,667)</td></tr>
  <tr><td>4</td><td>Count Dooku (10)</td><td>Mace Windu (24)</td><td>Ki-Adi-Mundi (0,063)</td><td>R2-D2 (0,263)</td><td>Mace Windu (0,593)</td></tr>
  <tr><td>5</td><td>Jar Jar (10)</td><td>Yoda (24)</td><td>Jar Jar (0,062)</td><td>C-3PO (0,177)</td><td>Yoda (0,593)</td></tr>
</table>
<p><i>Nota: en el grado hay un empate a 10 contactos entre Count Dooku, Jar Jar, Mace Windu y Yoda. Mace Windu y Yoda también empatan en grado ponderado y en cercanía.</i></p>

<p><b>El protagonista.</b> Anakin encabeza las cinco métricas. Comparte escena con 21 de los otros 32 personajes (un 66 %) y acumula 65 escenas compartidas. A diferencia del episodio I, en el que Qui-Gon destacaba claramente sobre el resto, aquí el protagonismo recae en un trío: Anakin, Obi-Wan y Padmé ocupan los tres primeros puestos en todas las métricas (solo se invierte el orden de Obi-Wan y Padmé en el autovector). La distancia con el cuarto puesto es muy grande: Padmé suma 50 escenas frente a las 24 de Mace Windu, y su intermediación es de 0,179 frente a 0,063. Qui-Gon, protagonista del episodio I, no interviene en este episodio.</p>

<p><b>Puentes entre grupos.</b> Anakin (0,383) y Obi-Wan (0,315) concentran la mayor parte de la intermediación, por delante de Padmé (0,179). Es coherente con una trama que se divide en dos líneas paralelas (la investigación de Obi-Wan y la protección de Padmé a cargo de Anakin), en la que ambos conectan los distintos escenarios. Fuera del trío destaca Ki-Adi-Mundi, que ocupa el cuarto puesto en intermediación (0,063) sin aparecer entre los diez primeros en ninguna otra métrica: en el grafo, Plo Koon queda conectado con el resto de la red a través de él.</p>

<p><b>Grado frente a grado ponderado.</b> Se repite el sesgo de reunión observado en el episodio I. Count Dooku (10 contactos), Jar Jar (10) y el Emperador (9) tienen un grado alto, pero su grado ponderado es bajo (16, 15 y 15 escenas respectivamente), y Bail Organa y Ask Aak (7 contactos cada uno) ni siquiera entran en el top 10 ponderado. Son personajes con muchos contactos pero pocas escenas con cada uno, típicos de escenas con muchos personajes como las del Senado o el Consejo Jedi. Mace Windu y Yoda tienen valores casi idénticos en todas las métricas, lo que indica que aparecen prácticamente siempre en las mismas escenas. En sentido contrario, R2-D2 (17) y C-3PO (15) entran en el top 10 ponderado sin estar entre los diez primeros por grado: hablan con pocos personajes, pero comparten muchas escenas con ellos. Además, ocupan el cuarto y quinto puesto en autovector (0,263 y 0,177), lo que indica que sus pocos contactos son con personajes muy centrales.</p>

<p><b>El papel del Emperador.</b> En el episodio II el Emperador (Palpatine) ocupa el noveno puesto en intermediación (0,028), muy por debajo del valor que tenía en el episodio I (0,100, sexto puesto), y no aparece entre los diez primeros en cercanía ni en autovector. Por tanto, tampoco en este episodio actúa como el nodo conector central de la red. Su papel se contrasta en el episodio III.</p>

<p><b>Comunidades.</b> El algoritmo detecta 8 comunidades, que se corresponden en buena medida con escenarios y facciones de la trama:</p>
<ul>
  <li><b>Senado y Consejo Jedi:</b> Orn Free Taa, Ask Aak, Bail Organa, Emperador y Jar Jar (Senado), junto con Yoda, Mace Windu, Ki-Adi-Mundi y Plo Koon (Consejo Jedi).</li>
  <li><b>Separatistas:</b> Count Dooku, Poggle, Nute Gunray y Sun Rit.</li>
  <li><b>Kamino:</b> Jango Fett, Boba Fett, Taun We y Lama Su.</li>
  <li><b>Familia de Padmé en Naboo:</b> Padmé, Jobal, Sola y Ruwee.</li>
  <li><b>Granja de los Lars en Tatooine:</b> Owen, Beru, Cliegg y C-3PO.</li>
  <li><b>Anakin y su pasado:</b> Anakin con Shmi, Watto y Sio Bibble.</li>
  <li><b>Obi-Wan:</b> Obi-Wan con Captain Typho y PK-4.</li>
  <li>R2-D2 queda como comunidad de un solo elemento.</li>
</ul>
<p>A diferencia del episodio I, en el que Qui-Gon, Anakin y Obi-Wan quedaban aislados en comunidades de un solo elemento, aquí Anakin y Obi-Wan sí quedan asignados a una comunidad. El único nodo aislado es R2-D2, cuyas conexiones se reparten entre varios grupos. En cambio, Padmé, que es el tercer personaje más central, queda dentro de la comunidad de su familia y no con los demás protagonistas.</p>

<p><b>Comparación con el episodio I.</b> La estructura de la red apenas cambia entre ambos episodios, pero sí cambia el protagonista (de Qui-Gon a Anakin) y la forma en que se reparte la centralidad: concentrada en un solo personaje en el episodio I y repartida entre un trío en el episodio II.</p>

<p>&nbsp;</p>

<h3>Análisis grafo episodio 3</h3>

<p>Grado</p>

In [71]:
grado = dict(G_ep3.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 14), ('OBI-WAN', 13), ('BAIL ORGANA', 12), ('EMPEROR', 11), ('PADME', 10), ('C-3PO', 9), ('YODA', 9), ('R2-D2', 8), ('MACE WINDU', 5), ('CLONE COMMANDER CODY', 4)]


<p>Grado ponderado.</p>

In [72]:
grado_pond = dict(G_ep3.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 83), ('OBI-WAN', 82), ('PADME', 40), ('BAIL ORGANA', 38), ('R2-D2', 38), ('EMPEROR', 34), ('YODA', 31), ('C-3PO', 26), ('MACE WINDU', 14), ('FANG ZAR', 7)]


<p>Intermediación (betweenness)</p>

In [73]:
inter = nx.betweenness_centrality(G_ep3)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('OBI-WAN', 0.2032561641257294), ('ANAKIN', 0.20154338415207984), ('EMPEROR', 0.19989648033126295), ('BAIL ORGANA', 0.19840956145303965), ('YODA', 0.12646339168078297), ('PADME', 0.10251270468661774), ('C-3PO', 0.04334650856389986), ('R2-D2', 0.024967061923583663), ('GENERAL GRIEVOUS', 0.012450592885375493), ('MACE WINDU', 0.007048748353096178)]


<p>Autovector (eigenvector)</p>

In [74]:
auto = nx.eigenvector_centrality(G_ep3, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 0.5596769514663992), ('OBI-WAN', 0.5543676352700888), ('R2-D2', 0.31033354567135846), ('EMPEROR', 0.26270793225647826), ('PADME', 0.2595752983942321), ('YODA', 0.22811007024030316), ('BAIL ORGANA', 0.21126564208825177), ('C-3PO', 0.16789520789719892), ('MACE WINDU', 0.11604165807090283), ('ODD BALL', 0.04553734784726006)]


<p>Cercanía (closeness)</p>

In [75]:
cerc = nx.closeness_centrality(G_ep3)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('ANAKIN', 0.71875), ('OBI-WAN', 0.696969696969697), ('BAIL ORGANA', 0.6764705882352942), ('EMPEROR', 0.6388888888888888), ('C-3PO', 0.6216216216216216), ('PADME', 0.6216216216216216), ('R2-D2', 0.6052631578947368), ('YODA', 0.5897435897435898), ('MACE WINDU', 0.5111111111111111), ('CAPTAIN TYPHO', 0.4791666666666667)]


<p>Comunidades</p>

In [76]:
comunidades = nx.community.greedy_modularity_communities(G_ep3, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

8
['JAR JAR', 'NUTE GUNRAY', 'COUNT DOOKU', 'GENERAL GRIEVOUS', 'EMPEROR', 'DARTH VADER']
['MACE WINDU', 'CLONE COMMANDER CODY', 'QUI-GON', 'CLONE COMMANDER GREE', 'YODA']
['BAIL ORGANA', 'MON MOTHMA', 'FANG ZAR', 'GIDDEAN DANU']
['CAPTAIN TYPHO', 'CAPTAIN ANTILLES', 'C-3PO']
['TION MEDON', 'ODD BALL', 'OBI-WAN']
['ANAKIN']
['PADME']
['R2-D2']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [77]:
print(
    f"nodos: {G_ep3.number_of_nodes()} aristas: {G_ep3.number_of_edges()}"
    f" densidad: {nx.density(G_ep3)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep3)}")

nodos: 24 aristas: 65 densidad: 0.23550724637681159
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [78]:
print(nx.average_shortest_path_length(G_ep3))
print(nx.diameter(G_ep3))

2.028985507246377
4


<h3>Resultados del episodio 3</h3>

<p><b>Estructura.</b> La red tiene 24 personajes y 65 aristas, con una densidad de 0,24. Es un grafo conexo (una sola componente), con un camino medio de 2,03 y un diámetro de 4. Es la red más pequeña y más densa de las analizadas hasta ahora (la densidad era de 0,19 en los episodios I y II): hay menos personajes y casi todos comparten escena con buena parte del resto.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Anakin (14)</td><td>Anakin (83)</td><td>Obi-Wan (0,203)</td><td>Anakin (0,560)</td><td>Anakin (0,719)</td></tr>
  <tr><td>2</td><td>Obi-Wan (13)</td><td>Obi-Wan (82)</td><td>Anakin (0,202)</td><td>Obi-Wan (0,554)</td><td>Obi-Wan (0,697)</td></tr>
  <tr><td>3</td><td>Bail Organa (12)</td><td>Padmé (40)</td><td>Emperador (0,200)</td><td>R2-D2 (0,310)</td><td>Bail Organa (0,676)</td></tr>
  <tr><td>4</td><td>Emperador (11)</td><td>Bail Organa (38)</td><td>Bail Organa (0,198)</td><td>Emperador (0,263)</td><td>Emperador (0,639)</td></tr>
  <tr><td>5</td><td>Padmé (10)</td><td>R2-D2 (38)</td><td>Yoda (0,126)</td><td>Padmé (0,260)</td><td>C-3PO (0,622)</td></tr>
</table>
<p><i>Nota: Bail Organa y R2-D2 empatan a 38 en grado ponderado, y C-3PO y Padmé empatan en cercanía (0,622).</i></p>

<p><b>Los protagonistas.</b> En el episodio III el protagonismo recae en un dúo casi perfectamente equilibrado: Anakin y Obi-Wan ocupan los dos primeros puestos en las cinco métricas (solo se invierte el orden en intermediación). Anakin comparte escena con 14 de los otros 23 personajes (un 61 %) y acumula 83 escenas compartidas, frente a las 82 de Obi-Wan. La diferencia entre ambos es mínima en todas las métricas, y la distancia con el tercero es enorme: Padmé, tercera en grado ponderado, suma 40 escenas, menos de la mitad que cada uno de los dos. Padmé pierde así el papel que tenía en el episodio II, donde formaba un trío con Anakin y Obi-Wan con valores similares (50 escenas frente a 65 y 51). Además, en este episodio Anakin se convierte en Darth Vader, que el dataset recoge como un nodo distinto y que no aparece en ningún ranking, por lo que la centralidad de Anakin no incluye su faceta de Vader.</p>

<p><b>Grado frente a grado ponderado.</b> Los dos rankings coinciden en los primeros puestos, pero hay matices. Bail Organa (12 contactos) y el Emperador (11) superan a Padmé (10) en grado, mientras que en grado ponderado Padmé los supera (40 escenas frente a 38 y 34), de modo que tienen más contactos pero menos escenas con cada uno. En sentido contrario, R2-D2 (8 contactos) y C-3PO (9) tienen menos contactos pero suman 38 y 26 escenas respectivamente. R2-D2 es además el tercero en autovector (0,310), por delante del Emperador y de Padmé, lo que indica que sus contactos son con los personajes más centrales de la red (Anakin y Obi-Wan), igual que ocurría en el episodio II.</p>

<p><b>Puentes entre grupos.</b> La intermediación está muy repartida: Obi-Wan (0,203), Anakin (0,202), el Emperador (0,200) y Bail Organa (0,198) tienen valores casi idénticos, y a continuación hay un salto claro hasta Yoda (0,126) y Padmé (0,103). Bail Organa destaca en particular: es tercero en grado y en cercanía, y cuarto en intermediación. Es, con diferencia, el personaje del Senado más central (los otros miembros de su comunidad, Mon Mothma, Fang Zar y Giddean Danu, apenas aparecen en los rankings), lo que indica que actúa como enlace entre ellos y el núcleo de la historia.</p>

<p><b>El papel del Emperador.</b> En el episodio III el Emperador (Palpatine) ocupa el tercer puesto en intermediación (0,200), a solo 0,003 de Obi-Wan y prácticamente igualado con Anakin. Es la mejor posición de los tres episodios analizados: su intermediación era de 0,100 en el episodio I (sexto puesto) y de 0,028 en el II (noveno). Además es cuarto en grado, en autovector y en cercanía. Lo más llamativo es que consigue esa capacidad de conexión con muchas menos escenas que los protagonistas: su grado ponderado es de 34, frente a las 83 y 82 de Anakin y Obi-Wan. Esto confirma en parte la hipótesis de que actúa como nodo conector entre facciones: en este episodio es uno de los cuatro principales puentes de la red, aunque no el primero. Encaja con su papel en la trama, en la que actúa a la vez ante los Jedi, el Senado y los separatistas.</p>

<p><b>Comunidades.</b> El algoritmo detecta 8 comunidades:</p>
<ul>
  <li><b>Sith y separatistas:</b> Emperador, Darth Vader, Count Dooku, General Grievous y Nute Gunray (junto con Jar Jar).</li>
  <li><b>Jedi y clones:</b> Yoda, Mace Windu, Clone Commander Cody, Clone Commander Gree y Qui-Gon.</li>
  <li><b>Senado:</b> Bail Organa, Mon Mothma, Fang Zar y Giddean Danu.</li>
  <li><b>Seguridad de Padmé y Bail Organa:</b> Captain Typho, Captain Antilles y C-3PO.</li>
  <li><b>Utapau:</b> Obi-Wan, Tion Medon y Odd Ball.</li>
  <li>Anakin, Padmé y R2-D2 quedan como comunidades de un solo elemento.</li>
</ul>
<p>Los tres nodos aislados son tres de los personajes más centrales de la red, y la razón es la misma que en los episodios anteriores: sus conexiones se reparten entre varios grupos, por lo que el algoritmo no puede asignarlos a ninguno. Anakin y Obi-Wan quedan además en comunidades distintas, algo coherente con una trama en la que ambos siguen líneas separadas durante buena parte del episodio. Darth Vader aparece agrupado con el Emperador, y no con Anakin, por tratarse de nodos distintos.</p>

<p><b>Comparación con los episodios anteriores.</b> La red se hace más compacta (densidad de 0,24 frente a 0,19) y la centralidad se concentra en dos personajes: Qui-Gon en el episodio I, un trío formado por Anakin, Obi-Wan y Padmé en el II, y un dúo formado por Anakin y Obi-Wan en el III. Mientras tanto, el Emperador pasa de ser un puente secundario a situarse entre los principales conectores de la red.</p>

<p>&nbsp;</p>

<h3>Análisis grafo episodio 4</h3>

<p>Grado</p>

In [79]:
grado = dict(G_ep4.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 15), ('LEIA', 12), ('C-3PO', 10), ('R2-D2', 9), ('BIGGS', 8), ('HAN', 8), ('RED LEADER', 7), ('OBI-WAN', 7), ('CHEWBACCA', 6), ('BERU', 5)]


<p>Grado ponderado.</p>

In [80]:
grado_pond = dict(G_ep4.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 128), ('HAN', 81), ('C-3PO', 63), ('LEIA', 56), ('R2-D2', 52), ('CHEWBACCA', 52), ('OBI-WAN', 44), ('BIGGS', 15), ('RED LEADER', 13), ('BERU', 10)]


<p>Intermediación (betweenness)</p>

In [81]:
inter = nx.betweenness_centrality(G_ep4)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 0.35632832080200516), ('LEIA', 0.25476190476190474), ('HAN', 0.19473684210526315), ('BIGGS', 0.04605263157894738), ('C-3PO', 0.04492481203007518), ('RED LEADER', 0.03684210526315789), ('R2-D2', 0.030451127819548875), ('OBI-WAN', 0.029448621553884714), ('DARTH VADER', 0.005263157894736842), ('BERU', 0.004699248120300752)]


<p>Autovector (eigenvector)</p>

In [82]:
auto = nx.eigenvector_centrality(G_ep4, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 0.552398339181853), ('HAN', 0.45237238491079673), ('C-3PO', 0.33262421176394374), ('CHEWBACCA', 0.32446924191914567), ('LEIA', 0.3178348884144225), ('R2-D2', 0.2912015947172406), ('OBI-WAN', 0.2827689839902239), ('BIGGS', 0.04888963037856285), ('BERU', 0.04459270632498222), ('OWEN', 0.0401436843191053)]


<p>Cercanía (closeness)</p>

In [83]:
cerc = nx.closeness_centrality(G_ep4)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 0.8), ('LEIA', 0.7142857142857143), ('C-3PO', 0.6666666666666666), ('R2-D2', 0.6451612903225806), ('HAN', 0.625), ('OBI-WAN', 0.6060606060606061), ('BIGGS', 0.5882352941176471), ('CHEWBACCA', 0.5882352941176471), ('RED LEADER', 0.5714285714285714), ('BERU', 0.5405405405405406)]


<p>Comunidades</p>

In [84]:
comunidades = nx.community.greedy_modularity_communities(G_ep4, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

8
['RED TEN', 'GOLD LEADER', 'CAMIE', 'RED LEADER', 'DODONNA', 'BIGGS', 'WEDGE']
['TARKIN', 'MOTTI', 'LEIA', 'DARTH VADER']
['R2-D2', 'BERU', 'OWEN']
['GREEDO', 'JABBA', 'HAN']
['C-3PO']
['LUKE']
['CHEWBACCA']
['OBI-WAN']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [85]:
print(
    f"nodos: {G_ep4.number_of_nodes()} aristas: {G_ep4.number_of_edges()}"
    f" densidad: {nx.density(G_ep4)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep4)}")

nodos: 21 aristas: 60 densidad: 0.2857142857142857
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [86]:
print(nx.average_shortest_path_length(G_ep4))
print(nx.diameter(G_ep4))

1.9142857142857144
3


<h3>Resultados del episodio 4</h3>

<p><b>Estructura.</b> La red tiene 21 personajes y 60 aristas, con una densidad de 0,29. Es un grafo conexo (una sola componente), con un camino medio de 1,91 y un diámetro de 3. Es la red más pequeña y más densa de las analizadas (la densidad era de 0,19 en los episodios I y II y de 0,24 en el III) y la primera con diámetro 3: desde cualquier personaje se llega a cualquier otro en un máximo de 3 pasos.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Luke (15)</td><td>Luke (128)</td><td>Luke (0,356)</td><td>Luke (0,552)</td><td>Luke (0,800)</td></tr>
  <tr><td>2</td><td>Leia (12)</td><td>Han (81)</td><td>Leia (0,255)</td><td>Han (0,452)</td><td>Leia (0,714)</td></tr>
  <tr><td>3</td><td>C-3PO (10)</td><td>C-3PO (63)</td><td>Han (0,195)</td><td>C-3PO (0,332)</td><td>C-3PO (0,667)</td></tr>
  <tr><td>4</td><td>R2-D2 (9)</td><td>Leia (56)</td><td>Biggs (0,046)</td><td>Chewbacca (0,324)</td><td>R2-D2 (0,645)</td></tr>
  <tr><td>5</td><td>Biggs (8)</td><td>R2-D2 (52)</td><td>C-3PO (0,045)</td><td>Leia (0,318)</td><td>Han (0,625)</td></tr>
</table>
<p><i>Nota: en el grado hay un empate a 8 contactos entre Biggs y Han, y en el grado ponderado empatan R2-D2 y Chewbacca (52).</i></p>

<p><b>El protagonista.</b> Luke encabeza las cinco métricas. Comparte escena con 15 de los otros 20 personajes (un 75 %) y acumula 128 escenas compartidas, frente a las 81 de Han, que es el segundo. Es el protagonismo más marcado de los cuatro episodios analizados: en grado ponderado el primero supera al segundo en un 58 %, frente al 36 % del episodio I, el 27 % del II y el 1 % del III. Su cercanía (0,800) es además la más alta de todos los protagonistas analizados hasta ahora (0,771 de Qui-Gon en el episodio I, 0,727 y 0,719 de Anakin en el II y el III).</p>

<p><b>Han y Leia.</b> Tras Luke, el segundo puesto depende de la métrica. Han es segundo en grado ponderado (81) y en autovector (0,452), y tercero en intermediación (0,195). Leia es segunda en grado (12), en intermediación (0,255) y en cercanía (0,714). Sus perfiles son distintos: Han concentra muchas escenas con pocos personajes (81 escenas con 8 contactos), mientras que Leia comparte menos escenas con más personajes (56 escenas con 12 contactos).</p>

<p><b>Grado frente a grado ponderado.</b> C-3PO es tercero en grado, grado ponderado, autovector y cercanía, y supera a Leia en escenas compartidas (63 frente a 56). Hay diferencias que ilustran de nuevo el sesgo de reunión: Biggs (8 contactos) y Red Leader (7) tienen un grado relativamente alto, pero suman solo 15 y 13 escenas (octavo y noveno puesto en grado ponderado). El autovector de Biggs es de solo 0,049, lo que indica que sus contactos son con personajes poco centrales, y son típicos de escenas de grupo como las de los pilotos. En sentido contrario, Chewbacca tiene pocos contactos (6, noveno por grado), pero suma 52 escenas (empatado con R2-D2 en el quinto puesto) y es cuarto en autovector (0,324), lo que indica que sus pocos contactos son con los personajes más centrales. Obi-Wan, que era uno de los dos personajes más centrales en los episodios II y III, queda aquí entre el sexto y el octavo puesto según la métrica (7 contactos, 44 escenas) y no aparece entre los cinco primeros de ninguna.</p>

<p><b>Puentes entre grupos.</b> Luke (0,356), Leia (0,255) y Han (0,195) concentran casi toda la intermediación de la red: el cuarto, Biggs, tiene solo 0,046. En el caso de Han, en el grafo Greedo y Jabba aparecen conectados con el resto de la red únicamente a través de él, y ambos forman con Han una comunidad. Leia, por su parte, comparte comunidad con Tarkin, Motti y Darth Vader, por lo que actúa de enlace entre el bando imperial y el resto de la red.</p>

<p><b>El bando imperial.</b> El Emperador no aparece en este episodio. El bando imperial está representado por Tarkin, Motti y Darth Vader, que quedan en la periferia de la red: ninguno de ellos entra entre los diez primeros en grado, grado ponderado, autovector o cercanía, y solo Darth Vader aparece en un ranking, el octavo puesto de intermediación con un valor casi nulo (0,005). Es el contraste con el episodio III, en el que el Emperador era uno de los principales puentes de la red.</p>

<p><b>Comunidades.</b> El algoritmo detecta 8 comunidades:</p>
<ul>
  <li><b>Pilotos rebeldes:</b> Red Leader, Red Ten, Gold Leader, Wedge y Dodonna, junto con Biggs y Camie.</li>
  <li><b>Bando imperial y Leia:</b> Tarkin, Motti, Darth Vader y Leia.</li>
  <li><b>Granja de Tatooine:</b> Owen, Beru y R2-D2.</li>
  <li><b>Mos Eisley:</b> Han, Greedo y Jabba.</li>
  <li>Luke, C-3PO, Chewbacca y Obi-Wan quedan como comunidades de un solo elemento.</li>
</ul>
<p>Los nodos aislados son personajes que comparten escena con varios grupos distintos, y por eso el algoritmo no puede asignarlos a ninguno. Luke y C-3PO son los dos más centrales de la red. Han y Leia, en cambio, sí quedan asignados a una comunidad, porque cada uno tiene personajes que dependen de ellos para conectarse con la red (Greedo y Jabba en el caso de Han, y el bando imperial en el de Leia).</p>

<p><b>Comparación con los episodios anteriores.</b> A lo largo de los cuatro episodios la red se hace cada vez más pequeña y densa: de 38 personajes y densidad 0,19 en el episodio I a 21 personajes y densidad 0,29 en el IV. La centralidad pasa de un único protagonista (Qui-Gon) a un trío (Anakin, Obi-Wan y Padmé), después a un dúo (Anakin y Obi-Wan) y vuelve a concentrarse en un solo personaje, Luke, con el mayor margen sobre el segundo de todos los episodios. Obi-Wan pasa de coprotagonista a personaje secundario.</p>

<p>&nbsp;</p>


<h3>Análisis grafo episodio 5</h3>

<p>Grado</p>

In [87]:
grado = dict(G_ep5.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 12), ('DARTH VADER', 11), ('HAN', 11), ('C-3PO', 10), ('LEIA', 10), ('CHEWBACCA', 9), ('LANDO', 8), ('BOBA FETT', 7), ('R2-D2', 6), ('PIETT', 4)]


<p>Grado ponderado.</p>

In [88]:
grado_pond = dict(G_ep5.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('HAN', 102), ('LEIA', 98), ('C-3PO', 93), ('CHEWBACCA', 81), ('LANDO', 42), ('LUKE', 35), ('R2-D2', 32), ('DARTH VADER', 25), ('PIETT', 11), ('BOBA FETT', 10)]


<p>Intermediación (betweenness)</p>

In [89]:
inter = nx.betweenness_centrality(G_ep5)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 0.4162178172704488), ('DARTH VADER', 0.32467532467532473), ('HAN', 0.11175894281157436), ('WEDGE', 0.09999999999999999), ('C-3PO', 0.07398951925267716), ('LEIA', 0.07398951925267716), ('CHEWBACCA', 0.026621098200045566), ('BOBA FETT', 0.02432216905901117), ('LANDO', 0.009778992936887672), ('PIETT', 0.007894736842105263)]


<p>Autovector (eigenvector)</p>

In [90]:
auto = nx.eigenvector_centrality(G_ep5, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('HAN', 0.513572457458832), ('LEIA', 0.5010988353477139), ('C-3PO', 0.47875163032165513), ('CHEWBACCA', 0.41676232312278066), ('LANDO', 0.2084300624859357), ('R2-D2', 0.1567682921878809), ('LUKE', 0.08692983177931603), ('RIEEKAN', 0.056837668281162566), ('DARTH VADER', 0.04558691201181766), ('BOBA FETT', 0.03159340895019112)]


<p>Cercanía (closeness)</p>

In [91]:
cerc = nx.closeness_centrality(G_ep5)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 0.7142857142857143), ('DARTH VADER', 0.6666666666666666), ('HAN', 0.6666666666666666), ('C-3PO', 0.6451612903225806), ('LEIA', 0.6451612903225806), ('CHEWBACCA', 0.625), ('LANDO', 0.6060606060606061), ('BOBA FETT', 0.5263157894736842), ('R2-D2', 0.5128205128205128), ('OBI-WAN', 0.46511627906976744)]


<p>Comunidades</p>

In [92]:
comunidades = nx.community.greedy_modularity_communities(G_ep5, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

9
['DACK', 'JANSON', 'ZEV', 'LUKE', 'WEDGE', 'OBI-WAN', 'YODA']
['BOBA FETT', 'OZZEL', 'PIETT', 'NEEDA', 'EMPEROR', 'DARTH VADER']
['DERLIN', 'C-3PO']
['CHEWBACCA']
['HAN']
['LANDO']
['LEIA']
['R2-D2']
['RIEEKAN']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [93]:
print(
    f"nodos: {G_ep5.number_of_nodes()} aristas: {G_ep5.number_of_edges()}"
    f" densidad: {nx.density(G_ep5)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep5)}")

nodos: 21 aristas: 55 densidad: 0.2619047619047619
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [94]:
print(nx.average_shortest_path_length(G_ep5))
print(nx.diameter(G_ep5))

2.0619047619047617
4


<h3>Resultados del episodio 5</h3>

<p><b>Estructura.</b> La red tiene 21 personajes y 55 aristas, con una densidad de 0,26. Es un grafo conexo (una sola componente), con un camino medio de 2,06 y un diámetro de 4. Tiene el mismo número de personajes que el episodio IV, pero con menos aristas (55 frente a 60), por lo que es algo menos densa (0,26 frente a 0,29) y el diámetro vuelve a ser 4. Sigue siendo más compacta que las redes de los episodios I, II y III.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Luke (12)</td><td>Han (102)</td><td>Luke (0,416)</td><td>Han (0,514)</td><td>Luke (0,714)</td></tr>
  <tr><td>2</td><td>Darth Vader (11)</td><td>Leia (98)</td><td>Darth Vader (0,325)</td><td>Leia (0,501)</td><td>Darth Vader (0,667)</td></tr>
  <tr><td>3</td><td>Han (11)</td><td>C-3PO (93)</td><td>Han (0,112)</td><td>C-3PO (0,479)</td><td>Han (0,667)</td></tr>
  <tr><td>4</td><td>C-3PO (10)</td><td>Chewbacca (81)</td><td>Wedge (0,100)</td><td>Chewbacca (0,417)</td><td>C-3PO (0,645)</td></tr>
  <tr><td>5</td><td>Leia (10)</td><td>Lando (42)</td><td>C-3PO (0,074)</td><td>Lando (0,208)</td><td>Leia (0,645)</td></tr>
</table>
<p><i>Nota: hay empates entre Darth Vader y Han (11 contactos y 0,667 de cercanía), entre C-3PO y Leia (10 contactos, 0,645 de cercanía) y entre C-3PO y Leia también en intermediación (0,074).</i></p>

<p><b>El protagonista depende de la métrica.</b> A diferencia de los episodios anteriores, en el episodio V no hay un único protagonista. Luke encabeza el grado (12 contactos, un 60 % de los otros 20 personajes), la intermediación (0,416) y la cercanía (0,714), pero en grado ponderado es solo sexto (35 escenas) y en autovector séptimo (0,087). Los primeros puestos en esas dos métricas los ocupan Han, Leia y C-3PO, casi igualados (102, 98 y 93 escenas, y 0,514, 0,501 y 0,479 de autovector), seguidos de Chewbacca (81 escenas, 0,417). Es el contraste con el episodio IV, en el que Luke encabezaba las cinco métricas con 128 escenas compartidas, frente a las 35 de este episodio. La interpretación es que la trama se divide en dos líneas: Luke destaca por estructura (con quién conecta y a cuántos grupos enlaza), mientras que el grupo formado por Han, Leia, C-3PO y Chewbacca destaca por el número de escenas que comparten entre sí y con el resto.</p>

<p><b>Grado frente a grado ponderado.</b> La diferencia entre ambas métricas explica este cambio. Los cuatro personajes del grupo de Han tienen entre 9 y 11 contactos y suman entre 81 y 102 escenas, es decir, entre 9 y 10 escenas por contacto. Luke, en cambio, tiene 12 contactos pero solo 35 escenas (menos de 3 por contacto): comparte escena con muchos personajes, pero pocas veces con cada uno. Darth Vader tiene un perfil parecido (11 contactos, 25 escenas, octavo en grado ponderado) y Boba Fett también (7 contactos, solo 10 escenas), lo que indica conexiones generadas por escenas de grupo. En sentido contrario, R2-D2 tiene solo 6 contactos pero suma 32 escenas, más que Darth Vader.</p>

<p><b>Puentes entre grupos.</b> La intermediación se concentra en dos personajes: Luke (0,416) y Darth Vader (0,325), con un salto claro hasta Han (0,112) y Wedge (0,100). Son los dos protagonistas del enfrentamiento principal del episodio, y cada uno actúa de enlace de un bloque distinto: Luke conecta a los pilotos, Yoda y Obi-Wan con el resto de la red, y Darth Vader conecta con el resto al bando imperial. Wedge destaca sin aparecer en las demás métricas: en el grafo, Janson queda conectado con el resto de la red a través de él.</p>

<p><b>El bando imperial.</b> Darth Vader es el único personaje del Imperio con peso en la red (segundo en grado, intermediación y cercanía), aunque con pocas escenas (25) y un autovector de solo 0,046. El Emperador no aparece en ningún ranking y, en el grafo, está unido únicamente a Darth Vader. Es una situación opuesta a la del episodio III, en el que el Emperador era uno de los principales puentes de la red: aquí es Darth Vader quien ocupa ese papel, aunque entre el bando imperial y los rebeldes.</p>

<p><b>Comunidades.</b> El algoritmo detecta 9 comunidades:</p>
<ul>
  <li><b>Luke, los pilotos y los maestros:</b> Luke, Wedge, Dack, Janson y Zev (pilotos), junto con Yoda y Obi-Wan.</li>
  <li><b>Flota imperial:</b> Darth Vader, el Emperador, Piett, Ozzel, Needa y Boba Fett.</li>
  <li><b>Derlin y C-3PO.</b></li>
  <li>Han, Leia, Chewbacca, Lando, R2-D2 y Rieekan quedan como comunidades de un solo elemento.</li>
</ul>
<p>Seis de las nueve comunidades tienen un solo elemento, la proporción más alta hasta ahora. Los personajes del grupo de Han (Han, Leia, Chewbacca) y Lando comparten escena con muchos personajes distintos, por lo que el algoritmo no puede asignarlos a ninguno. En cambio, Luke sí queda dentro de una comunidad, con los personajes que dependen de él para conectarse con la red, igual que ocurría con Han y Leia en el episodio IV.</p>

<p><b>Comparación con los episodios anteriores.</b> La red mantiene el tamaño del episodio IV (21 personajes), pero la centralidad ya no se concentra en un único personaje: se divide entre quien ocupa el centro de la estructura (Luke) y el grupo que acumula más escenas (Han, Leia, C-3PO y Chewbacca). C-3PO se mantiene entre los tres primeros en grado ponderado y autovector, igual que en el episodio IV. Aparece además por primera vez un antagonista con una intermediación alta (Darth Vader, 0,325), que sustituye al papel del Emperador en el episodio III.</p>

<p>&nbsp;</p>

<h3>Análisis grafo episodio 6</h3>

<p>Grado</p>

In [95]:
grado = dict(G_ep6.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 15), ('C-3PO', 11), ('HAN', 9), ('LANDO', 9), ('LEIA', 9), ('MON MOTHMA', 8), ('R2-D2', 8), ('ADMIRAL ACKBAR', 7), ('CHEWBACCA', 7), ('DARTH VADER', 5)]


<p>Grado ponderado.</p>

In [96]:
grado_pond = dict(G_ep6.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('HAN', 86), ('C-3PO', 80), ('LEIA', 72), ('CHEWBACCA', 72), ('LUKE', 69), ('R2-D2', 56), ('LANDO', 32), ('DARTH VADER', 13), ('ADMIRAL ACKBAR', 11), ('JABBA', 9)]


<p>Intermediación (betweenness)</p>

In [97]:
inter = nx.betweenness_centrality(G_ep6)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 0.5968810916179337), ('DARTH VADER', 0.20760233918128654), ('C-3PO', 0.06423001949317737), ('LANDO', 0.058089668615984406), ('ADMIRAL ACKBAR', 0.04327485380116959), ('HAN', 0.024853801169590642), ('LEIA', 0.024853801169590642), ('JABBA', 0.018031189083820662), ('R2-D2', 0.009941520467836256), ('BOUSHH', 0.0029239766081871343)]


<p>Autovector (eigenvector)</p>

In [98]:
auto = nx.eigenvector_centrality(G_ep6, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('HAN', 0.4769259421895332), ('C-3PO', 0.4312410639160178), ('LEIA', 0.4196164867877518), ('CHEWBACCA', 0.4180913025635771), ('R2-D2', 0.32739655843815924), ('LUKE', 0.32527926329083967), ('LANDO', 0.12692926689700662), ('JABBA', 0.03993137521768207), ('MON MOTHMA', 0.038663220219176155), ('ADMIRAL ACKBAR', 0.035305821580982)]


<p>Cercanía (closeness)</p>

In [99]:
cerc = nx.closeness_centrality(G_ep6)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('LUKE', 0.8260869565217391), ('C-3PO', 0.6551724137931034), ('HAN', 0.6129032258064516), ('LANDO', 0.6129032258064516), ('LEIA', 0.6129032258064516), ('MON MOTHMA', 0.59375), ('R2-D2', 0.59375), ('ADMIRAL ACKBAR', 0.5757575757575758), ('CHEWBACCA', 0.5757575757575758), ('DARTH VADER', 0.5428571428571428)]


<p>Comunidades</p>

In [100]:
comunidades = nx.community.greedy_modularity_communities(G_ep6, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

8
['JERJERROD', 'ANAKIN', 'PIETT', 'EMPEROR', 'DARTH VADER']
['MON MOTHMA', 'WEDGE', 'ADMIRAL ACKBAR', 'LANDO']
['BOUSHH', 'JABBA', 'C-3PO', 'BIB FORTUNA']
['LUKE', 'YODA', 'OBI-WAN']
['HAN']
['LEIA']
['CHEWBACCA']
['R2-D2']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [101]:
print(
    f"nodos: {G_ep6.number_of_nodes()} aristas: {G_ep6.number_of_edges()}"
    f" densidad: {nx.density(G_ep6)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep6)}")

nodos: 20 aristas: 55 densidad: 0.2894736842105263
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [102]:
print(nx.average_shortest_path_length(G_ep6))
print(nx.diameter(G_ep6))

1.9473684210526316
4


<h3>Resultados del episodio 6</h3>

<p><b>Estructura.</b> La red tiene 20 personajes y 55 aristas, con una densidad de 0,29. Es un grafo conexo (una sola componente), con un camino medio de 1,95 y un diámetro de 4. Es la red más pequeña de las analizadas hasta ahora y, por muy poco, la más densa (0,2895 frente a 0,2857 del episodio IV), aunque el diámetro es de 4 pasos, uno más que en el episodio IV.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Luke (15)</td><td>Han (86)</td><td>Luke (0,597)</td><td>Han (0,477)</td><td>Luke (0,826)</td></tr>
  <tr><td>2</td><td>C-3PO (11)</td><td>C-3PO (80)</td><td>Darth Vader (0,208)</td><td>C-3PO (0,431)</td><td>C-3PO (0,655)</td></tr>
  <tr><td>3</td><td>Han (9)</td><td>Leia (72)</td><td>C-3PO (0,064)</td><td>Leia (0,420)</td><td>Han (0,613)</td></tr>
  <tr><td>4</td><td>Lando (9)</td><td>Chewbacca (72)</td><td>Lando (0,058)</td><td>Chewbacca (0,418)</td><td>Lando (0,613)</td></tr>
  <tr><td>5</td><td>Leia (9)</td><td>Luke (69)</td><td>Admiral Ackbar (0,043)</td><td>R2-D2 (0,327)</td><td>Leia (0,613)</td></tr>
</table>
<p><i>Nota: Han, Lando y Leia empatan a 9 contactos y a 0,613 de cercanía, y Leia y Chewbacca empatan a 72 escenas en grado ponderado.</i></p>

<p><b>El protagonista depende de la métrica.</b> Igual que en el episodio V, no hay un único protagonista. Luke encabeza el grado (15 contactos, un 79 % de los otros 19 personajes), la intermediación (0,597) y la cercanía (0,826), pero es quinto en grado ponderado (69 escenas) y sexto en autovector (0,325). Han es primero en grado ponderado (86 escenas) y en autovector (0,477). La diferencia con el episodio V es que Luke se acerca al grupo de Han en escenas compartidas: pasa de 35 a 69 escenas, y la distancia con el primero en grado ponderado es de 17 escenas (en el episodio V era de 67). Los cinco primeros en grado ponderado (Han, C-3PO, Leia, Chewbacca y Luke) quedan entre 69 y 86 escenas.</p>

<p><b>Grado frente a grado ponderado.</b> De nuevo, el contraste entre ambas métricas explica la diferencia. Han, C-3PO, Leia y Chewbacca suman entre 7 y 10 escenas por contacto, mientras que Luke suma menos de 5 (69 escenas con 15 contactos): comparte escena con casi todos los personajes, pero pocas veces con cada uno. C-3PO es el más constante del grupo: es segundo en grado, grado ponderado, autovector y cercanía, tras haber sido tercero en grado ponderado y autovector en los episodios IV y V. Se repite también el sesgo de reunión: Mon Mothma (8 contactos) y Admiral Ackbar (7) tienen un grado relativamente alto, pero Mon Mothma no entra en el top 10 ponderado, Ackbar suma solo 11 escenas y ambos tienen un autovector inferior a 0,04. En sentido contrario, R2-D2 tiene 8 contactos pero suma 56 escenas y es quinto en autovector (0,327).</p>

<p><b>Puentes entre grupos.</b> La intermediación se concentra en dos personajes: Luke (0,597) y Darth Vader (0,208), con un salto muy claro hasta C-3PO (0,064), Lando (0,058) y Admiral Ackbar (0,043). El valor de Luke es el más alto de todos los episodios analizados (el máximo anterior era el suyo propio en el episodio V, con 0,416), y también lo es su cercanía (0,826, por delante de los 0,800 del episodio IV). Luke actúa de enlace entre su comunidad (Yoda y Obi-Wan) y el resto de la red. Lando y Admiral Ackbar, ambos de la comunidad del mando rebelde, son puentes secundarios.</p>

<p><b>El bando imperial.</b> Darth Vader es el único personaje del Imperio con peso en la red: tiene solo 5 contactos (décimo por grado) y 13 escenas, pero es segundo en intermediación (0,208), lo que indica que el resto de su comunidad depende de él para conectarse. Ocupa el mismo papel que en el episodio V, aunque con menos intermediación (0,325 entonces). El Emperador, a pesar de ser el antagonista final, no aparece entre los diez primeros de ninguna métrica. Su intermediación solo fue relevante en el episodio III. En este episodio aparecen además como nodos separados Anakin y Darth Vader, y ambos quedan agrupados en la misma comunidad.</p>

<p><b>Comunidades.</b> El algoritmo detecta 8 comunidades:</p>
<ul>
  <li><b>Bando imperial:</b> Darth Vader, el Emperador, Piett, Jerjerrod y Anakin.</li>
  <li><b>Mando rebelde:</b> Mon Mothma, Admiral Ackbar, Wedge y Lando.</li>
  <li><b>Palacio de Jabba:</b> Jabba, Bib Fortuna, Boushh y C-3PO.</li>
  <li><b>Luke y sus maestros:</b> Luke, Yoda y Obi-Wan.</li>
  <li>Han, Leia, Chewbacca y R2-D2 quedan como comunidades de un solo elemento.</li>
</ul>
<p>Cuatro de las ocho comunidades tienen un solo elemento, y son los personajes que comparten escena con muchos grupos distintos. A diferencia del episodio V, Luke queda asignado a una comunidad, y lo hace con los personajes que dependen de él para conectarse con la red. Llama la atención que C-3PO, el segundo personaje más central, quede dentro de la comunidad del palacio de Jabba, por la que pasa con muchas escenas compartidas.</p>

<p><b>Comparación con los episodios anteriores.</b> El episodio VI mantiene el patrón del V: Luke es el centro de la estructura de la red (grado, intermediación y cercanía), mientras que el grupo de Han lidera en escenas compartidas y en autovector. Así, en la trilogía original Luke es el protagonista por estructura y Han por número de escenas, con la diferencia de que en el episodio IV Luke lideraba todas las métricas. La concentración de la intermediación en el protagonista aumenta a lo largo de la trilogía (0,356, 0,416 y 0,597) y el antagonista (Darth Vader) mantiene el segundo puesto en los episodios V y VI.</p>

<p>&nbsp;</p>

<h3>Análisis grafo episodio 7</h3>

<p>Grado</p>

In [103]:
grado = dict(G_ep7.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('POE', 16), ('FINN', 14), ('HAN', 14), ('BB-8', 12), ('CHEWBACCA', 12), ('C-3PO', 10), ('LEIA', 10), ('REY', 10), ('KYLO REN', 10), ('SNAP', 8)]


<p>Grado ponderado.</p>

In [104]:
grado_pond = dict(G_ep7.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('FINN', 113), ('HAN', 98), ('REY', 87), ('CHEWBACCA', 81), ('BB-8', 76), ('POE', 46), ('LEIA', 27), ('KYLO REN', 25), ('C-3PO', 24), ('MAZ', 13)]


<p>Intermediación (betweenness)</p>

In [105]:
inter = nx.betweenness_centrality(G_ep7)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('POE', 0.2991103341103341), ('KYLO REN', 0.2212443112443112), ('REY', 0.11921511821511821), ('BB-8', 0.08970085470085469), ('FINN', 0.08241314241314245), ('HAN', 0.08241314241314245), ('GENERAL HUX', 0.08153846153846153), ('CHEWBACCA', 0.07342168942168942), ('CAPTAIN PHASMA', 0.0561915861915862), ('C-3PO', 0.022794871794871795)]


<p>Autovector (eigenvector)</p>

In [106]:
auto = nx.eigenvector_centrality(G_ep7, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('FINN', 0.5133445080990346), ('HAN', 0.4510262403700517), ('REY', 0.44950471167687545), ('CHEWBACCA', 0.4091091621025656), ('BB-8', 0.3368211849101521), ('POE', 0.1423981374711407), ('LEIA', 0.09627809242043804), ('KYLO REN', 0.07597240639600496), ('MAZ', 0.07301947424392237), ('C-3PO', 0.06886248023680451)]


<p>Cercanía (closeness)</p>

In [107]:
cerc = nx.closeness_centrality(G_ep7)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('POE', 0.6842105263157895), ('FINN', 0.6666666666666666), ('HAN', 0.6666666666666666), ('CHEWBACCA', 0.6341463414634146), ('KYLO REN', 0.6190476190476191), ('BB-8', 0.5777777777777777), ('LEIA', 0.5531914893617021), ('REY', 0.5531914893617021), ('CAPTAIN PHASMA', 0.5531914893617021), ('C-3PO', 0.5416666666666666)]


<p>Comunidades</p>

In [108]:
comunidades = nx.community.greedy_modularity_communities(G_ep7, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

9
['SNAP', 'YOLO ZIFF', 'NIV LEK', 'ADMIRAL ACKBAR', 'POE', 'ELLO ASTY', 'JESS', 'LEIA', 'ADMIRAL STATURA']
['GENERAL HUX', 'CAPTAIN PHASMA', 'COLONEL DATOO', 'KYLO REN', 'LIEUTENANT MITAKA', 'SNOKE', 'LOR SAN TEKKA']
['UNKAR PLUTT', 'BB-8', 'BALA-TIK']
['R2-D2', 'C-3PO']
['LUKE', 'REY']
['FINN']
['HAN']
['CHEWBACCA']
['MAZ']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [109]:
print(
    f"nodos: {G_ep7.number_of_nodes()} aristas: {G_ep7.number_of_edges()}"
    f" densidad: {nx.density(G_ep7)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep7)}")

nodos: 27 aristas: 92 densidad: 0.2621082621082621
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [110]:
print(nx.average_shortest_path_length(G_ep7))
print(nx.diameter(G_ep7))

2.0826210826210825
4


<h3>Resultados del episodio 7</h3>


<h3>Resultados del episodio 7</h3>

<p><b>Estructura.</b> La red tiene 27 personajes y 92 aristas, con una densidad de 0,26. Es un grafo conexo (una sola componente), con un camino medio de 2,08 y un diámetro de 4. La red vuelve a crecer tras los episodios IV, V y VI (21, 21 y 20 personajes), y su densidad es prácticamente igual a la del episodio V (0,262).</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Poe (16)</td><td>Finn (113)</td><td>Poe (0,299)</td><td>Finn (0,513)</td><td>Poe (0,684)</td></tr>
  <tr><td>2</td><td>Finn (14)</td><td>Han (98)</td><td>Kylo Ren (0,221)</td><td>Han (0,451)</td><td>Finn (0,667)</td></tr>
  <tr><td>3</td><td>Han (14)</td><td>Rey (87)</td><td>Rey (0,119)</td><td>Rey (0,450)</td><td>Han (0,667)</td></tr>
  <tr><td>4</td><td>BB-8 (12)</td><td>Chewbacca (81)</td><td>BB-8 (0,090)</td><td>Chewbacca (0,409)</td><td>Chewbacca (0,634)</td></tr>
  <tr><td>5</td><td>Chewbacca (12)</td><td>BB-8 (76)</td><td>Finn (0,082)</td><td>BB-8 (0,337)</td><td>Kylo Ren (0,619)</td></tr>
</table>
<p><i>Nota: Finn y Han empatan en grado (14), en cercanía (0,667) y en intermediación (0,082, quinto puesto compartido). BB-8 y Chewbacca empatan a 12 contactos.</i></p>

<p><b>El protagonista depende de la métrica.</b> Poe encabeza el grado (16 contactos, un 62 % de los otros 26 personajes), la intermediación (0,299) y la cercanía (0,684), pero es sexto en grado ponderado (46 escenas) y en autovector (0,142). Finn, en cambio, encabeza el grado ponderado (113 escenas) y el autovector (0,513), y es segundo en grado y en cercanía. Finn y Han son los más constantes, ya que ambos están entre los cinco primeros en las cinco métricas. Rey nunca es la primera, pero es tercera en grado ponderado, intermediación y autovector. Poe lidera así tres métricas y Finn dos, el reparto más disperso de todos los episodios analizados.</p>

<p><b>Grado frente a grado ponderado.</b> Finn, Rey, Han, Chewbacca y BB-8, los cinco primeros en grado ponderado, suman entre 6 y 9 escenas por contacto. Poe, Leia, Kylo Ren y C-3PO, en cambio, tienen 10 o más contactos pero menos de 3 escenas por contacto (46, 27, 25 y 24 escenas respectivamente), y sus autovectores son bajos (entre 0,069 y 0,142): comparten escena con muchos personajes, pero pocas veces con cada uno, algo típico de escenas de grupo como las de la Resistencia. C-3PO, que estaba entre los más centrales en la trilogía original, cae al noveno puesto en grado ponderado y al décimo en autovector.</p>

<p><b>Puentes entre grupos.</b> La intermediación se concentra en Poe (0,299) y Kylo Ren (0,221), con un salto claro hasta Rey (0,119), BB-8 (0,090) y Finn y Han (0,082). Poe conecta con el resto de la red a la comunidad de la Resistencia. Kylo Ren, con solo 25 escenas y un autovector de 0,076, es el segundo puente de la red, por lo que su papel es estructural y no depende del número de escenas, como el de Darth Vader en los episodios V y VI. Además, el General Hux (séptimo, 0,082) y Captain Phasma (noveno, 0,056) también están entre los diez primeros en intermediación: la Primera Orden tiene tres personajes con papel de puente, frente a uno solo (Darth Vader) del Imperio en el episodio VI.</p>

<p><b>Luke.</b> Luke no aparece en ningún ranking y, en el grafo, está conectado únicamente con Rey, con quien forma comunidad.</p>

<p><b>Comunidades.</b> El algoritmo detecta 9 comunidades:</p>
<ul>
  <li><b>Resistencia:</b> Poe, Leia, Snap, Yolo Ziff, Niv Lek, Admiral Ackbar, Ello Asty, Jess y Admiral Statura.</li>
  <li><b>Primera Orden:</b> Kylo Ren, General Hux, Captain Phasma, Snoke, Colonel Datoo, Lieutenant Mitaka y Lor San Tekka.</li>
  <li><b>Jakku:</b> BB-8, Unkar Plutt y Bala-Tik.</li>
  <li><b>Los droides:</b> R2-D2 y C-3PO.</li>
  <li><b>Luke y Rey.</b></li>
  <li>Finn, Han, Chewbacca y Maz quedan como comunidades de un solo elemento.</li>
</ul>
<p>Cuatro de las nueve comunidades tienen un solo elemento, y son personajes que comparten escena con varios grupos distintos, por lo que el algoritmo no puede asignarlos a ninguno. Las demás comunidades se corresponden con bandos y escenarios del episodio. Leia queda dentro de la comunidad de Poe y la Resistencia, y no con Han y Chewbacca.</p>

<p><b>Comparación con los episodios anteriores.</b> En el episodio VII la centralidad se reparte entre personajes nuevos (Poe, Finn, Rey y BB-8) y dos veteranos de la trilogía original (Han y Chewbacca, que siguen entre los cuatro primeros en grado ponderado y autovector), mientras que Leia, C-3PO y Luke pierden el papel que tenían en la trilogía. Se repite, además, el patrón de los episodios V y VI: el protagonista por estructura (Poe, como Luke antes) no coincide con el protagonista por escenas (Finn, como Han), y un personaje del bando antagonista (Kylo Ren) ocupa el segundo puesto en intermediación.</p>

<p>&nbsp;</p>

<h3>Análisis grafo general (todos los episodios)</h3>

<p>Grado</p>

In [111]:
grado = dict(G_general.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('DARTH VADER', 52), ('OBI-WAN', 36), ('C-3PO', 35), ('PADME', 34), ('QUI-GON', 27), ('HAN', 26), ('LUKE', 26), ('LEIA', 25), ('JAR JAR', 24), ('EMPEROR', 22)]


<p>Grado ponderado.</p>

In [112]:
grado_pond = dict(G_general.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('HAN', 360), ('C-3PO', 310), ('DARTH VADER', 309), ('CHEWBACCA', 297), ('R2-D2', 279), ('OBI-WAN', 261), ('LEIA', 258), ('LUKE', 241), ('PADME', 173), ('QUI-GON', 152)]


<p>Intermediación (betweenness)</p>

In [113]:
inter = nx.betweenness_centrality(G_general)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('DARTH VADER', 0.28953479206513344), ('OBI-WAN', 0.16566867063854887), ('C-3PO', 0.14164337519692108), ('LUKE', 0.13337008200734393), ('HAN', 0.10593551781744437), ('POE', 0.07434475042119036), ('CHEWBACCA', 0.06940205808510326), ('EMPEROR', 0.06530321032604398), ('PADME', 0.06329791061081946), ('LEIA', 0.05968298498131624)]


<p>Autovector (eigenvector)</p>

In [114]:
auto = nx.eigenvector_centrality(G_general, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('HAN', 0.4662379552577977), ('CHEWBACCA', 0.40572509385903016), ('C-3PO', 0.40340439028509406), ('LEIA', 0.3897047081659043), ('LUKE', 0.31305943982383), ('R2-D2', 0.3109530105003732), ('OBI-WAN', 0.16862232756134743), ('DARTH VADER', 0.15495006147486784), ('LANDO', 0.11263304818487391), ('FINN', 0.10446153904252817)]


<p>Cercanía (closeness)</p>

In [115]:
cerc = nx.closeness_centrality(G_general)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('DARTH VADER', 0.6123595505617978), ('C-3PO', 0.5647668393782384), ('OBI-WAN', 0.5618556701030928), ('HAN', 0.5343137254901961), ('LUKE', 0.529126213592233), ('LEIA', 0.5190476190476191), ('R2-D2', 0.5190476190476191), ('CHEWBACCA', 0.5117370892018779), ('PADME', 0.5093457943925234), ('EMPEROR', 0.5046296296296297)]


<p>Comunidades</p>

In [116]:
comunidades = nx.community.greedy_modularity_communities(G_general, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

18
['YOLO ZIFF', 'SNAP', 'GENERAL HUX', 'NIV LEK', 'ADMIRAL ACKBAR', 'POE', 'COLONEL DATOO', 'ELLO ASTY', 'CAPTAIN PHASMA', 'JESS', 'FINN', 'LIEUTENANT MITAKA', 'KYLO REN', 'SNOKE', 'LOR SAN TEKKA', 'ADMIRAL STATURA']
['TEY HOW', 'DOFINE', 'SIO BIBBLE', 'SUN RIT', 'NUTE GUNRAY', 'RUNE', 'COUNT DOOKU', 'DARTH MAUL', 'GENERAL GRIEVOUS', 'EMPEROR', 'POGGLE', 'GENERAL CEEL']
['JAR JAR', 'SENATOR ASK AAK', 'CAPTAIN TYPHO', 'FODE/BEED', 'SEBULBA', 'JABBA', 'BIB FORTUNA', 'BOUSHH', 'ORN FREE TAA', 'TARPALS', 'SHMI']
['DACK', 'RED TEN', 'GOLD LEADER', 'JANSON', 'ZEV', 'LUKE', 'CAMIE', 'RED LEADER', 'DODONNA', 'BIGGS', 'WEDGE']
['TAUN WE', 'BOBA FETT', 'ODD BALL', 'LAMA SU', 'PK-4', 'OBI-WAN', 'JANGO FETT', 'TC-14', 'TION MEDON']
['OWEN', 'CLIEGG', 'SOLA', 'JOBAL', 'PADME', 'BERU', 'RUWEE']
['RABE', 'MACE WINDU', 'KI-ADI-MUNDI', 'PLO KOON', 'CLONE COMMANDER CODY', 'CLONE COMMANDER GREE', 'YODA']
['CAPTAIN ANTILLES', 'MON MOTHMA', 'FANG ZAR', 'GIDDEAN DANU', 'BAIL ORGANA', 'VALORUM']
['GREEDO', 

<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [117]:
print(
    f"nodos: {G_general.number_of_nodes()} aristas: {G_general.number_of_edges()}"
    f" densidad: {nx.density(G_general)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_general)}")

nodos: 110 aristas: 444 densidad: 0.07406171809841534
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [118]:
print(nx.average_shortest_path_length(G_general))
print(nx.diameter(G_general))

2.6100083402835694
6


<h3>Resultados del grafo general (todos los episodios)</h3>

<p><b>Estructura.</b> La red tiene 110 personajes y 444 aristas, con una densidad de 0,074. Es un grafo conexo (una sola componente), con un camino medio de 2,61 y un diámetro de 6. Es mucho menos densa que cualquiera de las redes de los episodios (entre 0,19 y 0,29) y tiene un diámetro mayor (6 frente a 3 o 4), algo esperable: los personajes de episodios distintos no comparten escena, así que la red global es un conjunto de redes de episodio unidas por los personajes que aparecen en varios de ellos.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Darth Vader (52)</td><td>Han (360)</td><td>Darth Vader (0,290)</td><td>Han (0,466)</td><td>Darth Vader (0,612)</td></tr>
  <tr><td>2</td><td>Obi-Wan (36)</td><td>C-3PO (310)</td><td>Obi-Wan (0,166)</td><td>Chewbacca (0,406)</td><td>C-3PO (0,565)</td></tr>
  <tr><td>3</td><td>C-3PO (35)</td><td>Darth Vader (309)</td><td>C-3PO (0,142)</td><td>C-3PO (0,403)</td><td>Obi-Wan (0,562)</td></tr>
  <tr><td>4</td><td>Padmé (34)</td><td>Chewbacca (297)</td><td>Luke (0,133)</td><td>Leia (0,390)</td><td>Han (0,534)</td></tr>
  <tr><td>5</td><td>Qui-Gon (27)</td><td>R2-D2 (279)</td><td>Han (0,106)</td><td>Luke (0,313)</td><td>Luke (0,529)</td></tr>
</table>
<p><i>Nota: Han y Luke quedan justo detrás del quinto puesto en grado, con 26 contactos.</i></p>

<p><b>El protagonista de la saga.</b> Como en los episodios V, VI y VII, el resultado depende de la métrica. Darth Vader encabeza el grado (52 contactos, un 48 % de los otros 109 personajes y casi el doble que Obi-Wan, que es el segundo), la intermediación (0,290) y la cercanía (0,612): es el centro de la red por estructura. Han, en cambio, encabeza el grado ponderado (360 escenas) y el autovector (0,466), y es el personaje que más escenas comparte con otros. Darth Vader concentra además las escenas de Anakin: en el grafo general ambos aparecen unificados en un único nodo, lo que explica su posición.</p>

<p><b>Grado frente a grado ponderado.</b> Han suma 360 escenas con solo 26 contactos (casi 14 escenas por contacto, la proporción más alta), y Chewbacca (297) y R2-D2 (279) están entre los cinco primeros en grado ponderado sin entrar entre los diez primeros en grado: hablan con pocos personajes, pero comparten muchas escenas con ellos. En sentido contrario, Padmé (34 contactos, 173 escenas), Qui-Gon (27 contactos, 152 escenas), Jar Jar (24) y el Emperador (22) tienen un grado alto pero un grado ponderado bajo, y los dos últimos ni siquiera entran en el top 10 ponderado. Es de nuevo el sesgo de reunión, por escenas con muchos personajes como las del Senado o el Consejo Jedi.</p>

<p><b>Autovector.</b> Los seis primeros en autovector (Han, Chewbacca, C-3PO, Leia, Luke y R2-D2, con valores entre 0,31 y 0,47) son personajes que aparecen en cuatro o más episodios, y hay un salto claro hasta el séptimo, Obi-Wan (0,169). Los pesos de las aristas de estos personajes se acumulan a lo largo de varios episodios, lo que favorece su centralidad en la red global.</p>

<p><b>Los personajes que unen los episodios.</b> Los cinco primeros en intermediación (Darth Vader, Obi-Wan, C-3PO, Luke y Han) son personajes que aparecen en varios episodios y actúan de enlace entre las redes de cada uno. Esto explica el diámetro de 6 de la red global. C-3PO es el más constante: es tercero en grado, intermediación y autovector, y segundo en grado ponderado y cercanía, y aparece (junto con R2-D2) en los siete episodios. Poe (sexto, 0,074) es el único de los diez primeros en intermediación que solo aparece en un episodio.</p>

<p><b>El papel del Emperador.</b> La hipótesis planteada en la propuesta del trabajo, que Palpatine actúa como nodo conector central entre facciones, no se confirma en el grafo general: el Emperador es décimo en grado (22) y en cercanía (0,505) y octavo en intermediación (0,065), muy por detrás de Darth Vader (0,290) y Obi-Wan (0,166). Solo en el episodio III llegó a situarse entre los principales puentes de su red (0,200, tercer puesto).</p>

<p><b>Comunidades.</b> El algoritmo detecta 18 comunidades, 13 con varios personajes y 5 de un solo elemento. Las primeras se corresponden con bandos y escenarios de episodios concretos:</p>
<ul>
  <li><b>Primera Orden y Resistencia (episodio VII):</b> Poe, Finn, Kylo Ren, General Hux, Captain Phasma, Snoke y Admiral Ackbar, junto con otros nueve personajes de ambos bandos (16 en total).</li>
  <li><b>Federación de Comercio, separatistas y Sith:</b> Nute Gunray, Count Dooku, Darth Maul, General Grievous, el Emperador, Poggle, Sun Rit, Rune, Dofine, Tey How, General Ceel y Sio Bibble.</li>
  <li><b>Carrera de vainas, gungans y palacio de Jabba:</b> Jar Jar, Tarpals, Sebulba, Fode/Beed, Jabba, Bib Fortuna, Boushh, Shmi, Captain Typho y los senadores Ask Aak y Orn Free Taa.</li>
  <li><b>Pilotos rebeldes:</b> Luke, Wedge, Biggs, Camie, Red Leader, Red Ten, Gold Leader, Dodonna, Dack, Janson y Zev.</li>
  <li><b>Kamino y Utapau:</b> Obi-Wan, Jango Fett, Boba Fett, Taun We, Lama Su, Odd Ball, Tion Medon, TC-14 y PK-4.</li>
  <li><b>Familias de Padmé y de los Lars:</b> Padmé, Jobal, Sola, Ruwee, Owen, Beru y Cliegg.</li>
  <li><b>Jedi y clones:</b> Yoda, Mace Windu, Ki-Adi-Mundi, Plo Koon, Clone Commander Cody, Clone Commander Gree y Rabe.</li>
  <li><b>Senado:</b> Bail Organa, Mon Mothma, Fang Zar, Giddean Danu, Captain Antilles y Valorum.</li>
  <li><b>Tatooine (episodio I):</b> Qui-Gon, Watto, Kitster, Wald, Greedo y Jira.</li>
  <li><b>Jakku:</b> Rey, BB-8, Maz, Unkar Plutt y Bala-Tik.</li>
  <li><b>Naboo (episodio I):</b> Boss Nass, Captain Panaka, Ric Olie, Bravo Two y Bravo Three.</li>
  <li><b>Flota imperial:</b> Darth Vader, Jerjerrod, Ozzel, Piett y Needa.</li>
  <li><b>Hoth y la Estrella de la Muerte:</b> Leia, Rieekan, Derlin, Tarkin y Motti.</li>
</ul>
<p>Las comunidades se organizan más por episodio y escenario que por personaje: cada una agrupa a los personajes que comparten escenas en un mismo bando o lugar. Los personajes que aparecen en varios episodios quedan asignados a la comunidad con la que más escenas comparten: Luke con los pilotos rebeldes (y no con Rey, que queda en Jakku), Leia con Hoth y la Estrella de la Muerte, Obi-Wan con Kamino y Utapau y Qui-Gon con Tatooine.</p>
<p>Los cinco nodos aislados son Han, Chewbacca, C-3PO, R2-D2 y Lando: personajes que comparten escena con varios de estos grupos y que por eso no pertenecen a ninguno. Cuatro de ellos (Han, C-3PO, Chewbacca y R2-D2) son además cuatro de los cinco primeros en grado ponderado. Darth Vader, en cambio, que es el personaje con más contactos (52), queda en una comunidad de solo cinco personajes, la flota imperial, por lo que la mayor parte de sus contactos, que incluyen los de Anakin, pertenecen a otras comunidades.</p>

<p><b>Síntesis.</b> No hay un único protagonista de la saga: depende de la métrica y del episodio. Por estructura, el centro de la red es Darth Vader (con Anakin), el personaje que más personajes conecta y por el que pasan más caminos mínimos. Por escenas compartidas, es Han. C-3PO es el personaje más constante. El Emperador solo destaca como puente en el episodio III. Por episodios, el protagonista pasa de Qui-Gon (I) a Anakin (II), a Anakin y Obi-Wan (III), a Luke (IV), al reparto entre Luke y Han (V y VI) y a Poe y Finn (VII), y en los episodios V, VI y VII un personaje antagonista (Darth Vader y Kylo Ren) ocupa el segundo puesto en intermediación.</p>

<p>&nbsp;</p>